# Home task: pandas 

## Question 1

- Load the energy data from the file [Energy Indicators.xls](http://unstats.un.org/unsd/environment/excel_file_tables/2013/Energy%20Indicators.xls).
It is a list of indicators of energy supply and renewable electricity production from the United Nations for the year 2013.


- It should be put into a DataFrame with the variable name of "energy"


- Make sure to exclude the footer and header information from the datafile.


- The first two columns are unneccessary, so you should get rid of them, and you should change the column labels so that the columns are:<br>
`['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']`


- Convert `Energy Supply` to gigajoules (there are 1,000,000 gigajoules in a petajoule).


- For all countries which have missing data (e.g. data with `...`) make sure this is reflected as `np.NaN` values.


- Rename the following list of countries (for use in later questions):
    - `Republic of Korea`: `South Korea`,
    - `United States of America`: `United States`,
    - `United Kingdom of Great Britain and Northern Ireland`: `United Kingdom`,
    - `China, Hong Kong Special Administrative Region`: `Hong Kong`


- There are also several countries with numbers and/or parenthesis in their name. Be sure to remove these, e.g.:
    - `Bolivia (Plurinational State of)` should be `Bolivia`,
    - `Switzerland17` should be `Switzerland`.


- Next, load the GDP data from the file ["world_bank.csv"](http://data.worldbank.org/indicator/NY.GDP.MKTP.CD). 
It is a csv containing countries' GDP from 1960 to 2015 from World Bank. Call this DataFrame "GDP"


- Make sure to skip the header, and rename the following list of countries:
    - `Korea, Rep.`: `South Korea`,
    - `Iran, Islamic Rep.`: `Iran`,
    - `Hong Kong SAR, China`: `Hong Kong`


- Finally, load the "Sciamgo Journal and Country Rank data for [Energy Engineering and Power Technology"](http://www.scimagojr.com/countryrank.php?category=2102). It ranks countries based on their journal contributions in the aforementioned area. Call this DataFrame "ScimEn"


- Join the three datasets: Energy, GDP, and ScimEn into a new dataset (using the intersection of country names). Use only the 10 years (2006-2015) of GDP data and only the top 15 countries by Scimagojr 'Rank' (Rank 1 through 15).


- The index of this DataFrame should be the name of the country, and the columns should be<br>
`['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', '% Renewable', '2006', '2007', '2008', '2009', '2010', '2011', 2012', '2013', '2014', '2015']`

Function "answer_one" should return the resulted DataFrame (20 columns and 15 entries)

## Answer the following questions in the context of only the top 15 countries by Scimagojr Rank (aka the DataFrame returned by `answer_one()`)

In [1]:
import pandas as pd
import numpy as np

In [2]:
def answer_one() -> pd.DataFrame:
    # Read only needed columns and convert '...' to NaN right away.
    energy = (
        pd.read_excel(
            'Energy Indicators.xls',
            skiprows=17,
            skipfooter=38,
            usecols=[2, 3, 4, 5],
            names=['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'],
            na_values='...')
            .assign(**{'Energy Supply': lambda df: df['Energy Supply'] * 1_000_000})
            .assign(
                # Clean country names: remove digits and text in brackets.
                Country=lambda df: (
                df["Country"]
                .str.replace(r"\d+|\s*\(.*?\)", "", regex=True)
                .str.strip())
                )
    )
    # Align country naming across datasets before merge.
    energy = energy.replace({'Country':{
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States',
        'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
        'China, Hong Kong Special Administrative Region': 'Hong Kong'
    }})

    GDP = (pd.read_csv('world_bank.csv', 
                       skiprows=4)
                      .rename(columns={'Country Name': 'Country'})
                      .replace({'Country':{
                                'Korea, Rep.': 'South Korea',
                                'Iran, Islamic Rep.': 'Iran',
                                'Hong Kong SAR, China': 'Hong Kong'}})
                      )
    
    # Keep only GDP values for 2006-2015.
    GDP = GDP[['Country', '2006', '2007', '2008',
                '2009', '2010', '2011', '2012',
                '2013', '2014', '2015']]

    # We need only top-15 countries by rank for the task.
    ScimEn_top_15 = (pd.read_excel('scimagojr_country_rank_1996-2024.xlsx')
                     .head(15)) #.query('Rank <= 15') 

    # Merge by country and set country names as index.
    return (
        ScimEn_top_15
        .merge(energy, how='inner', on='Country')
        .merge(GDP, how='inner', on='Country')
        .set_index('Country')
    )
    
answer_one()

    

,Rank,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,,
China,1,Asiatic Region,472465,470142,6591474,4594673,13.95,380,1.271910e+11,93.0,...,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551546e+12,8.532185e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,Northern America,222772,217929,4131736,1119917,18.55,491,9.083800e+10,286.0,...,1.381558e+13,1.447423e+13,1.476986e+13,1.447807e+13,1.504897e+13,1.559973e+13,1.625397e+13,1.688068e+13,1.760814e+13,1.829502e+13
India,3,Asiatic Region,96975,94714,1243636,479638,12.82,258,3.319500e+10,26.0,...,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856722e+12,2.039126e+12,2.103588e+12
United Kingdom,4,Western Europe,61946,60287,1326766,207337,21.42,313,7.920000e+09,124.0,...,2.708442e+12,3.090510e+12,2.929412e+12,2.412840e+12,2.485483e+12,2.663806e+12,2.707090e+12,2.784854e+12,3.064708e+12,2.927911e+12
Japan,5,Asiatic Region,61939,61307,813472,166907,13.13,246,1.898400e+10,149.0,...,4.601663e+12,4.579751e+12,5.106679e+12,5.289493e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
Germany,6,Western Europe,55466,54296,932195,185376,16.81,274,1.326100e+10,165.0,...,3.046309e+12,3.484057e+12,3.808786e+12,3.479801e+12,3.468154e+12,3.824829e+12,3.597897e+12,3.808086e+12,3.965801e+12,3.423568e+12
Russian Federation,7,Eastern Europe,49938,49562,268391,112125,5.37,123,3.070900e+10,214.0,...,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,Northern America,44877,44004,1082361,161133,24.12,305,1.043100e+10,296.0,...,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,Western Europe,42635,40695,767280,171442,18.00,227,6.530000e+09,109.0,...,1.958564e+12,2.222524e+12,2.417508e+12,2.209484e+12,2.144936e+12,2.306974e+12,2.097929e+12,2.153226e+12,2.173256e+12,1.845428e+12


### Question 2
What is the average GDP over the last 10 years for each country? (exclude missing values from this calculation.)

*This function should return a Series named `avgGDP` with 15 countries and their average GDP sorted in descending order.*

In [3]:
def answer_two() -> pd.Series:
    # Slice GDP years directly by labels, then compute average per country.
    return (
        answer_one()
        .loc[:, '2006':'2015']
        .mean(axis=1)
        .sort_values(ascending=False)
        .rename('AvgGDP')
    )
    
answer_two()

Country
United States         1.572243e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.590729e+12
United Kingdom        2.777505e+12
France                2.692000e+12
Italy                 2.152983e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.406644e+12
South Korea           1.221328e+12
Australia             1.207997e+12
Iran                  4.567516e+11
Name: AvgGDP, dtype: float64

### Question 3
By how much had the GDP changed over the 10 year span for the country with the 6th largest average GDP?

*This function should return a single number.*

In [4]:
def answer_three() -> float:
    top15       = answer_one()
    # Take country with 6th largest average GDP.
    country_6th = answer_two().index[5]

    return top15.loc[country_6th, '2015'] - top15.loc[country_6th, '2006']

answer_three()

np.float64(124621907951.68018)

### Question 4

Create a new column that is the ratio of Self-Citations to Total Citations. 
What is the maximum value for this new column, and what country has the highest ratio?

*This function should return a tuple with the name of the country and the ratio.*

In [5]:
def answer_four() -> tuple[str, float]:
    # Ratio of self-citations to total citations for each country.
    citation_ratios = (
        answer_one()
        .assign(**{
            "Citation Ratio": lambda df: df["Self-citations"] / df["Citations"]})
        ["Citation Ratio"]
    )
    country = citation_ratios.idxmax()
    return country, citation_ratios.max()
answer_four()

('China', np.float64(0.6970630544852335))

### Question 5

Create a column that estimates the population using Energy Supply and Energy Supply per capita. 
What is the third most populous country according to this estimate?

*This function should return a single string value.*

In [6]:
def answer_five() -> str:
    return (
        answer_one()
        .assign(Population=lambda df: df['Energy Supply'] / df['Energy Supply per Capita'])
        .sort_values(by='Population', ascending=False)
        .index[2]
    )

answer_five()

'United States'

### Question 6
Create a column that estimates the number of citable documents per person. 
What is the correlation between the number of citable documents per capita and the energy supply per capita? Use the `.corr()` method, (Pearson's correlation).

*This function should return a single number.*


In [7]:
def answer_six() -> float:
    return (
        answer_one()
        .assign(Population=lambda df: df["Energy Supply"] / df["Energy Supply per Capita"])
        # Estimate documents per person, then correlate with energy per capita.
        .assign(**{ "Docs per Capita": lambda df: df["Citable documents"] / df["Population"] })
        [["Docs per Capita", "Energy Supply per Capita"]]
        .corr().loc["Docs per Capita", "Energy Supply per Capita"]
    )
answer_six()

np.float64(0.6905473831164102)

### Question 7
Use the following dictionary to group the Countries by Continent, then create a dateframe that displays the sample size (the number of countries in each continent bin), and the sum, mean, and std deviation for the estimated population of each country.

```python
ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
```

*This function should return a DataFrame with index named Continent `['Asia', 'Australia', 'Europe', 'North America', 'South America']` and columns `['size', 'sum', 'mean', 'std']`*

In [8]:
def answer_seven() -> pd.DataFrame:

    ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
    
    return (
        answer_one()
        .assign(Population=lambda df: df['Energy Supply'] / df['Energy Supply per Capita'])
        # Group countries by continent and summarize estimated population.
        .groupby(ContinentDict)['Population']
        .agg(['size', 'sum', 'mean', 'std'])
        .rename_axis('Continent')
    )

answer_seven()

,size,sum,mean,std
Continent,,,,
Asia,5,2.898666e+09,5.797333e+08,6.790979e+08
Australia,1,2.331602e+07,2.331602e+07,NaN
Europe,6,4.579297e+08,7.632161e+07,3.464767e+07
North America,2,3.528552e+08,1.764276e+08,1.996696e+08
South America,1,2.059153e+08,2.059153e+08,NaN
